In [0]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as f
import pyarrow.parquet as pq
from torch.utils.data import IterableDataset, DataLoader
import pickle
import os
import json

In [0]:
# ============================================================
# Parquet Streaming Dataset
# ============================================================
class RankingDataset(IterableDataset):
    def __init__(self, parquet_dir, 
                 user_id_col, product_id_col,
                 user_cat_cols, user_num_cols,
                 product_cat_cols, product_num_cols,
                 batch_size=1024):
        super().__init__()
        self.user_id_col, self.product_id_col = user_id_col, product_id_col
        self.user_cat_cols, self.user_num_cols = user_cat_cols, user_num_cols
        self.product_cat_cols, self.product_num_cols = product_cat_cols, product_num_cols
        self.batch_size = batch_size

        self.parquet_files = list(Path(parquet_dir).glob("*.parquet"))

    def __iter__(self):
        for parquet_file in self.parquet_files:
            pf = pq.ParquetFile(parquet_file)
            for batch in pf.iter_batches(batch_size=self.batch_size):
                pdf = batch.to_pandas()
                user_id = torch.tensor(pdf[self.user_id_col].values, dtype=torch.long)
                product_id = torch.tensor(pdf[self.product_id_col].values, dtype=torch.long)

                user_cat = torch.stack([torch.tensor(pdf[c].values, dtype=torch.long) for c in self.user_cat_cols], dim=1)
                user_num = torch.stack([torch.tensor(pdf[c].values, dtype=torch.float32) for c in self.user_num_cols], dim=1)

                product_cat = torch.stack([torch.tensor(pdf[c].values, dtype=torch.long) for c in self.product_cat_cols], dim=1)
                product_num = torch.stack([torch.tensor(pdf[c].values, dtype=torch.float32) for c in self.product_num_cols], dim=1)         

                labels = {
                    "detail_viewed": torch.tensor(pdf["label_detail_viewed"].values, dtype=torch.float32),
                    "order_initiated": torch.tensor(pdf["label_order_initiated"].values, dtype=torch.float32),
                    "order_placed": torch.tensor(pdf["label_order_placed"].values, dtype=torch.float32),
                    "payment": torch.tensor(pdf["label_payment"].fillna(0).values, dtype=torch.float32),
                }
                yield {"user_id": user_id,
                        "user_cat": user_cat,
                        "user_num": user_num,
                        "product_id": product_id,
                        "product_cat": product_cat,
                        "product_num": product_num,
                        "labels": labels}            

In [0]:
# ============================================================
# Model: shared trunk + heads
# ============================================================
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used to process concatenated feature vectors.
    Consists of several linear layers with ReLU activations and dropout for regularization.
    """
    def __init__(self, in_dim, hidden_dims=(32,16), dropout=0.2):
        super().__init__()
        layers = []  # List to hold layers
        prev = in_dim  # Track input dimension for each layer
        for h in hidden_dims:
            # Add a Linear layer, followed by ReLU and Dropout
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h  # Update input dimension for next layer
        self.net = nn.Sequential(*layers)  # Combine layers into a sequential model

    def forward(self, x):
        # Forward pass through the MLP
        return self.net(x)

def _last_linear_out_features(seq):
    """
    Utility to get output dimension of last Linear layer in a Sequential.
    """
    for l in reversed(seq):
        if isinstance(l, nn.Linear): return l.out_features
    raise ValueError("No Linear found")

def calibrate_probability(p_pred, alpha: float):
    """
    Calibrate probability after training based on negative downsampling.
    Args:
        p_pred: raw sigmoid probability (tensor)
        alpha: global downsampling ratio (0 < alpha < 1)
    Returns:
        Calibrated probability tensor
    """
    return (alpha * p_pred) / ((1 - p_pred) + alpha * p_pred)

class MultiTaskRanker(nn.Module):
    def __init__(self, uid_vocab, uid_dim, user_cat_dims, user_num_dim,
                 pid_vocab, pid_dim, product_cat_dims, product_num_dim, 
                 hidden_dims=(32,16), dropout=0.2, alpha=1):
        super().__init__()
        # Embedding for user ID (with padding index 0)
        self.user_id_emb = nn.Embedding(uid_vocab+1, uid_dim, padding_idx=0)
        # Embeddings for each categorical user feature
        self.user_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in user_cat_dims])

        # Embedding for product ID (with padding index 0)
        self.product_id_emb = nn.Embedding(pid_vocab+1, pid_dim, padding_idx=0)
        # Embeddings for each categorical product feature
        self.product_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in product_cat_dims])

        u_dim = uid_dim + sum(d for _, d in user_cat_dims) + user_num_dim
        p_dim = pid_dim + sum(d for _, d in product_cat_dims) + product_num_dim

        # MLP to combine all features into a single vector
        self.mlp = MLP(u_dim+p_dim, hidden_dims, dropout)

        last_size = _last_linear_out_features(self.mlp.net)
        self.head_detail_viewed = nn.Linear(last_size, 1)
        self.head_order_initiated = nn.Linear(last_size, 1)
        self.head_order_placed = nn.Linear(last_size, 1)
        # payment head predicts z, with exp(z) ≈ payment
        self.head_payment = nn.Linear(last_size, 1)

        self.alpha = alpha  # global downsampling ratio

    def forward(self, user_id, user_cat, user_num, product_id, product_cat, product_num, output_type='logit'):
        # Embed user IDs
        u_id = self.user_id_emb(user_id)
        # Embed and concatenate all categorical features (if any)
        if self.user_cat_embs:
            u_cat = torch.cat([emb(user_cat[:, i]) for i, emb in enumerate(self.user_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            u_cat = torch.zeros(u_id.size(0), 0, device=u_id.device)

        # Embed product ID
        p_id = self.product_id_emb(product_id)
        # Embed and concatenate all categorical features (if any)
        if self.product_cat_embs:
            p_cat = torch.cat([emb(product_cat[:, i]) for i, emb in enumerate(self.product_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            p_cat = torch.zeros(p_id.size(0), 0, device=p_id.device)

        # Concatenate all features and pass through MLP
        h = self.mlp(torch.cat([u_id, u_cat, user_num, p_id, p_cat, product_num], dim=-1))

        if output_type == 'logit':
            # Raw logit outputs
            detail_viewed_pred = self.head_detail_viewed(h).squeeze(-1)
            order_initiated_pred = self.head_order_initiated(h).squeeze(-1)
            order_placed_pred = self.head_order_placed(h).squeeze(-1)
            # dwell time: transformation base on [P Covington, et.al. Deep Neural Networks for YouTube Recommendations. In RecSys, 2016]
            payment_pred = self.head_payment(h).squeeze(-1)
        elif output_type == 'prob':
            # Raw probablity outputs
            detail_viewed_pred = torch.sigmoid(self.head_detail_viewed(h).squeeze(-1))
            order_initiated_pred = torch.sigmoid(self.head_order_initiated(h).squeeze(-1))
            order_placed_pred = torch.sigmoid(self.head_order_placed(h).squeeze(-1))
            payment_pred = torch.sigmoid(self.head_payment(h).squeeze(-1))
        elif output_type == 'calibrated_prob':
            # Apply calibration to probablities when downsampling applied to imbalanced class
            # Xinran He et al. Practical lessons from predicting clicks on ads at Facebook. In the 8th International Workshop on Data Mining for Online Advertising.
            detail_viewed_pred = calibrate_probability(torch.sigmoid(self.head_detail_viewed(h).squeeze(-1)), self.alpha)
            order_initiated_pred = calibrate_probability(torch.sigmoid(self.head_order_initiated(h).squeeze(-1)), self.alpha)
            order_placed_pred = calibrate_probability(torch.sigmoid(self.head_order_placed(h).squeeze(-1)), self.alpha)
            payment_pred = torch.sigmoid(self.head_payment(h).squeeze(-1))  # no calibration for payment
        else:
            ValueError("Output_type must be logit, prob or calibrated_prob")

        return {
            "detail_viewed": detail_viewed_pred,
            "order_initiated": order_initiated_pred,
            "order_placed": order_placed_pred,
            "payment": payment_pred,
        }

In [0]:
# -----------------------------
# Training loop
# -----------------------------
class RankerTrainer:
    """
    Enhanced trainer that supports multiple negative sampling strategies.
    Supports different objectives and negative sampling strategies.
    """
    def __init__(self, model, lr=1e-3, device="cpu"):
        self.model = model.to(device)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.device = device

    def _bce_loss_fn(self, output_type='logit', pos_weight=None):
        if output_type=='logit':
            return nn.BCEWithLogitsLoss(pos_weight = pos_weight) 
        else:
            return nn.BCELoss()
        
    def _payment_loss_fn(self, logits: torch.Tensor, true_payment: torch.Tensor):
        """
        Custom loss: if t = true payment,
        y = t/(1+t), p = sigmoid(z), loss = -(t*log(p) + log(1-p)).
        """
        p = torch.sigmoid(logits)
        t = true_payment
        loss = -(t * torch.log(p + 1e-9) + torch.log(1 - p + 1e-9))
        return loss.mean()

    def _batch_to_device(self, batch):
        # Moves batch tensors to the target device
        td = lambda x: x.to(self.device)
        feats = (td(batch['user_id']), td(batch['user_cat']), td(batch['user_num']),
                td(batch['product_id']), td(batch['product_cat']), td(batch['product_num']))
        
        labels = batch['labels']
        labs = (td(labels['detail_viewed']), td(labels['order_initiated']), td(labels['order_placed']), td(labels['payment']))

        return feats, labs    

    def train_one_epoch(self, loader, output_type='logit', task_weights={k:1 for k in ('detail_viewed','order_initiated','order_placed','payment')}, pos_weights=None):
        self.model.train()
        total_loss = 0.0
        
        n_batches = 0
        for batch in loader:
            features, labels = self._batch_to_device(batch)
            preds = self.model(*features, output_type=output_type)
            
            lab_detail_viewed, lab_order_initiated, lab_order_placed, lab_payment = labels

            if not pos_weights:
                # When no predefined positive-negative class ratio presents, use in-batch sample to estimate the class imbalance ratio
                pos_weights={
                    'detail_viewed':torch.sum(lab_detail_viewed==0)/torch.sum(lab_detail_viewed==1),
                    'order_initiated':torch.sum(lab_order_initiated==0)/torch.sum(lab_order_initiated==1),
                    'order_placed':torch.sum(lab_order_placed==0)/torch.sum(lab_order_placed==1)
                }

            loss_fn_detail_viewed = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('detail_viewed', 1)]))
            loss_fn_order_initiated = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('order_initiated', 1)]))
            loss_fn_order_placed = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('order_placed', 1)]))
            loss_fn_payment = self._bce_loss_fn(pos_weight=None)

            self.opt.zero_grad()
            loss = 0.0
            loss += task_weights.get('detail_viewed',1.0) * loss_fn_detail_viewed(preds['detail_viewed'], lab_detail_viewed)
            loss += task_weights.get('order_initiated',1.0) * loss_fn_order_initiated(preds['order_initiated'], lab_order_initiated)
            loss += task_weights.get('order_placed',1.0) * loss_fn_order_placed(preds['order_placed'], lab_order_placed)
            # loss += task_weights.get('payment', 1.0) * self._payment_loss_fn(preds['payment'], lab_payment)
            # transform lab_payment: y = t/(1+t)
            loss += task_weights.get('payment', 1.0) * loss_fn_payment(preds['payment'], lab_payment/(1 + lab_payment))

            loss.backward()
            self.opt.step()
            total_loss += loss.item()
            n_batches += 1

        return total_loss / n_batches

#### Run training pipeline

In [0]:
# Get pipeline parameters
model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
user_id_col = 'user_id'
product_id_col = 'product_id'

user_cat_cols = ["platform"]
user_num_cols=['store_visits_30d',
'store_visits_90d',
'pdp_views_30d',
'pdp_views_90d',
'tot_items_ordered_30d',
'tot_items_ordered_90d',
'tot_items_ordered_180d',
'app_items_ordered_30d',
'app_items_ordered_90d',
'app_items_ordered_180d',
'coupons_owned_30d',
'coupons_owned_90d',
'coupons_owned_180d',
'coupons_redeemed_30d',
'coupons_redeemed_90d',
'coupons_redeemed_180d']

product_cat_cols = ["product_type"]
product_num_cols=['card_uv_imps_30d',
'card_uv_imps_90d',
'pdp_uv_imps_30d',
'pdp_uv_imps_90d',
'tot_order_cnt_30d',
'tot_order_cnt_90d',
'app_order_cnt_30d',
'app_order_cnt_90d',
'avg_order_msrp',
'avg_order_baseprice_60d',
'avg_trans_price_30d',
'avg_trans_price_90d']

label_cols = ['label_detail_viewed', 'label_order_initiated', 'label_order_placed', 'label_payment']

cat_cols = user_cat_cols + product_cat_cols
num_cols = user_num_cols + product_num_cols

user_feature_cols = user_cat_cols + user_num_cols
product_feature_cols = product_cat_cols + product_num_cols

parquet_dir = model_config['TRAIN_PARQUET_PATH']
metadata_dir = model_config['METADATA_PATH']
model_dir = model_config['MODEL_PATH']

neg_ratio = model_config['neg_sample_ratio']
batch_size = model_config['batch_size']

In [0]:
metadata = pickle.load(open(Path(metadata_dir) / 'metadata_stats.pkl', 'rb'))
class_ratios = pickle.load(open(Path(metadata_dir) / 'class_ratios.pkl', 'rb'))
print("Data stats:")
print(f"Number of users: {metadata['n_users']}")
print(f"Number of products: {metadata['n_products']}")
print(f"user categorical dims: {metadata['user_cat_dims']}")
print(f"product categorical dims: {metadata['product_cat_dims']}")
print(f"user numerical dim: {metadata['user_num_dim']}")
print(f"product numerical dim: {metadata['product_num_dim']}")
class_ratios_display = {k.replace('label_', ''): v for k, v in class_ratios.items()}
print(f"Class ratios: {class_ratios_display}")

In [0]:
# Create datasets
train_dataset = RankingDataset(
    parquet_dir=parquet_dir,
    user_id_col=user_id_col+'_idx', 
    product_id_col=product_id_col+'_idx',
    user_cat_cols=[c+'_idx' for c in user_cat_cols],
    user_num_cols=[c+'_norm' for c in user_num_cols],
    product_cat_cols=[c+'_idx' for c in product_cat_cols],
    product_num_cols=[c+'_norm' for c in product_num_cols],
    batch_size=batch_size
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=None, num_workers=0)

print(f"Train dataset: {len(train_dataset.parquet_files)} parquet files")

In [0]:
model_args = model_config['model_args']
user_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['user_cat_dims']]
product_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['product_cat_dims']]

model = MultiTaskRanker(uid_vocab=metadata['n_users'], 
                        uid_dim=model_args['id_emb_dim'],
                        user_cat_dims=user_cat_dims,
                        user_num_dim=metadata['user_num_dim'],
                        pid_vocab=metadata['n_products'], 
                        pid_dim=model_args['id_emb_dim'],
                        product_cat_dims=product_cat_dims,
                        product_num_dim=metadata['product_num_dim'],
                        hidden_dims=model_args['hidden_dims'], 
                        dropout=model_args['dropout'], 
                        alpha=neg_ratio)

In [0]:
def train_streaming(trainer, train_loader, output_type='logit', task_weights={k:1 for k in ('detail_viewed','order_initiated','order_placed','payment')}, pos_weights=None, epochs=5):
    """
    Training loop specifically for streaming datasets.
    """
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(1, epochs+1):
        # Training
        loss = trainer.train_one_epoch(train_loader, output_type, task_weights, pos_weights)
        
        print(f"Epoch {epoch}:")
        print(f"  Training Loss: {loss:.4f}")

    return trainer.model  

trainer = RankerTrainer(model, lr=model_config['trainer_args']['lr'], device="cpu")
output_type = model_config['output_type']
task_weights = model_config['task_weights']
pos_weights = {k: 1/v for k,v in class_ratios.items()}
epochs = model_config['epochs']

model = train_streaming(
        trainer,
        train_loader,
        output_type,
        task_weights,
        pos_weights,
        epochs=epochs
    )

In [0]:
model_path = Path(model_dir).joinpath('model')
torch.save(model.state_dict(), model_path)